# Azure RAG

In [1]:
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizedQuery
from dotenv import load_dotenv
import os
import truststore
import uuid
import re

load_dotenv()

True

# SSL

In [2]:
'''
Subject == Issuer 
So that PEM is self-signed and really is a “root” cert.
The remaining problem is exactly what the error says
In that certificate, the Basic Constraints extension isnt marked critical
The OpenSSL/libssl stack your Python is using is enforcing that rule when its used as a CA trust anchor.
'''

os.environ.pop("REQUESTS_CA_BUNDLE", None)
os.environ.pop("SSL_CERT_FILE", None)
truststore.inject_into_ssl()

# Load Env Variables

In [3]:
azure_endpoint = os.getenv("AZURE_ENDPOINT")
api_key = os.getenv("API_KEY")
api_version = os.getenv("API_VERSION")
deployment_name = os.getenv("DEPLOYMENT_NAME")

embeddings_endpoint = os.getenv("EMBEDDINGS_ENDPOINT")
embeddings_api_key = os.getenv("EMBEDDINGS_API_KEY")
embeddings_api_version = os.getenv("EMBEDDINGS_API_VERSION")
embeddings_deployment_name = os.getenv("EMBEDDINGS_DEPLOYMENT_NAME")

ai_search_endpoint = os.getenv("AI_SEARCH_ENDPOINT")
ai_search_index = os.getenv("AI_SEARCH_INDEX")
ai_search_api_key = os.getenv("AI_SEARCH_API_KEY")

# Clients

In [4]:
client_llm = AzureOpenAI(
    azure_endpoint = azure_endpoint,
    api_key = api_key,
    api_version = api_version
)

In [5]:
client_embeddings = AzureOpenAI(
    azure_endpoint = embeddings_endpoint,
    api_key = embeddings_api_key,
    api_version = embeddings_api_version
)

In [6]:
client_search = SearchClient(
    endpoint = ai_search_endpoint,
    index_name = ai_search_index,
    credential = AzureKeyCredential(ai_search_api_key)
)

In [7]:
client_index = SearchIndexClient(
    endpoint = ai_search_endpoint, 
    credential = AzureKeyCredential(ai_search_api_key)
)

idx = client_index.get_index(ai_search_index)
print("All fields:", [f.name for f in idx.fields])

All fields: ['id', 'content', 'contentVector']


# Text =  Wizard of Oz Book

In [8]:
# Load Book
file_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/Stuff/wizard_of_oz.txt"
with open(file_path, "r") as file:
    full_text = file.read()

# Remove Introduction
book_start = full_text.find("Chapter I.")
book = full_text[book_start:]
len(book)

# Remove Illustrations
oz_book = re.sub(r'\[Illustration:.*?\]', '\n', book, flags=re.DOTALL)
len(oz_book)

204542

# Split into chunks

In [9]:
def split_into_chunks(text, chunk_size=1000, chunk_overlap=100):
    
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - chunk_overlap
        # Starting position is moved forward by chunk size minus overlap to create overlapping chunks

    return chunks

# Embed Chunks

In [10]:
def create_embeddings(chunks):

    embeddings = client_embeddings.embeddings.create(
        model = embeddings_deployment_name,
        input = chunks
    )

    embeddings = [item.embedding for item in embeddings.data]
    
    return embeddings

# Create Document Dictionaries

In [11]:
def create_documents(chunks, embeddings):

    documents = []
    for i in range(len(chunks)):
        documents.append({
            "id": str(uuid.uuid4()),
            "content": chunks[i], 
            "contentVector": embeddings[i]
        })

    return documents

# Upload to Azure

In [12]:
def upload_documents(documents):

    result = client_search.upload_documents(documents)

    return result

# Embed Query

In [13]:
query = "What color are the main character's shoes?"
query_vec = create_embeddings([query])[0]

# Vector Search

In [14]:
def vector_search(query_vec, knn=5, top_vectors=3):

    vector_search_params = VectorizedQuery(
        vector = query_vec,
        k_nearest_neighbors = knn,
        fields = "contentVector"
    )

    search_results = client_search.search(
        search_text = None,  # Vector Search Only
        vector_queries = [vector_search_params],
        select = ["id", "content"],
        top = top_vectors
    )

    return list(search_results)

# Create Context from Retrieved Chunks

In [15]:
def create_context(retrieved_results):

    retrieved_chunks = []
    for r in retrieved_results:
        retrieved_chunks.append(r["content"])
        
    retrieved_context = "\n\n".join(retrieved_chunks)
    return retrieved_context

# Create Prompt

In [16]:
def create_prompt(retrieved_context, query):

    prompt = f"""

    Instructions: You are a helpful assistant. 
    Use only the context provided to answer the question.
    If you don't know the answer, say you don't know. Do not try to make up an answer.

    Context: {retrieved_context}

    Question: {query}

    """
    
    return prompt

# Use LLM

In [17]:
def llm_response(prompt, temperature=0, max_tokens=200):

    messages = [{"role": "user", "content": prompt}]

    completion = client_llm.chat.completions.create(
        model = deployment_name,
        messages = messages,
        max_tokens = 200,
        temperature = 0
    )

    return completion.choices[0].message.content

# Delete Documents From Azure Index

In [18]:
def delete_documents():

    doc_count = client_search.get_document_count()

    results = client_search.search(
        search_text="*",
        select=["id"],
        top=doc_count
    )

    ids = [{"id": doc["id"]} for doc in results]
    if len(ids) > 0:
        results_deleted = client_search.delete_documents(ids)
        print(f"Deleted all {len(ids)} documents.")
    else:
        print("No documents to delete.")

# Upload Documents to Azure Flow
1. Split into Chunks
2. Embed Chunks
3. Create Documents
4. Upload Documents


In [19]:
chunks = split_into_chunks(oz_book, chunk_size=1000, chunk_overlap=100)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 228


In [20]:
embeddings = create_embeddings(chunks)
print(f"Total embeddings created: {len(embeddings)}")

Total embeddings created: 228


In [21]:
documents = create_documents(chunks, embeddings)
print(f"Total documents created: {len(documents)}")

Total documents created: 228


In [22]:
uploaded = upload_documents(documents)

In [23]:
# len(chunks) == len(embeddings) == client_search.get_document_count()

# RAG Flow
1. Embed Query
2. Vector Search
3. Create Context
4. Create Prompt
5. LLM Response

In [24]:
query = "What color are the main character's shoes?"
query_vec = create_embeddings([query])[0]

In [25]:
retrieved_results = vector_search(query_vec, knn=5, top_vectors=3)
print(f"Total retrieved results: {len(retrieved_results)}")

Total retrieved results: 3


In [26]:
retrieved_context = create_context(retrieved_results)

In [27]:
prompt = create_prompt(retrieved_context, query)
print(prompt)



    Instructions: You are a helpful assistant. 
    Use only the context provided to answer the question.
    If you don't know the answer, say you don't know. Do not try to make up an answer.

    Context: the middle of the kitchen floor, and then by her magic arts made
the iron invisible to human eyes. So that when Dorothy walked across
the floor she stumbled over the bar, not being able to see it, and
fell at full length. She was not much hurt, but in her fall one of
the Silver Shoes came off, and before she could reach it the Witch
had snatched it away and put it on her own skinny foot.
The wicked woman was greatly pleased with the success of her trick,
for as long as she had one of the shoes she owned half the power of
their charm, and Dorothy could not use it against her, even had she
known how to do so.

The little girl, seeing she had lost one of her pretty shoes, grew
angry, and said to the Witch,
"Give me back my shoe!"
"I will not," retorted the Witch, "for it is now my sh

In [28]:
response = llm_response(prompt, temperature=0, max_tokens=200)
print(response)

The main character's shoes are silver.


# Delete Documents

In [29]:
delete_documents()

Deleted all 228 documents.
